In [ ]:
#-----NOT IN USE (Scrape by State filter)

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC
import time
from bs4 import BeautifulSoup
import pandas as pd
from selenium.common.exceptions import TimeoutException, NoSuchElementException
import os

# -----------------------------
# Static List of States
# -----------------------------
STATES = {
    "29": "KARNATAKA"
}

# STATES = {
#     "35": "ANDAMAN AND NICOBAR ISLANDS", "28": "ANDHRA PRADESH", "12": "ARUNACHAL PRADESH", 
#     "18": "ASSAM", "10": "BIHAR", "4": "CHANDIGARH", "22": "CHHATTISGARH", "7": "DELHI", 
#     "30": "GOA", "24": "GUJARAT", "6": "HARYANA", "2": "HIMACHAL PRADESH", "1": "JAMMU AND KASHMIR", 
#     "20": "JHARKHAND", "29": "KARNATAKA", "32": "KERALA", "37": "LADAKH", "31": "LAKSHADWEEP", 
#     "23": "MADHYA PRADESH", "27": "MAHARASHTRA", "14": "MANIPUR", "17": "MEGHALAYA", "15": "MIZORAM", 
#     "13": "NAGALAND", "21": "ODISHA", "34": "PUDUCHERRY", "3": "PUNJAB", "8": "RAJASTHAN", 
#     "11": "SIKKIM", "33": "TAMIL NADU", "36": "TELANGANA", 
#     "38": "THE DADRA AND NAGAR HAVELI AND DAMAN AND DIU", "16": "TRIPURA", "5": "UTTARAKHAND", 
#     "9": "UTTAR PRADESH", "19": "WEST BENGAL"
# }

driver = webdriver.Chrome()
driver.get("https://parivesh.nic.in/newupgrade/#/trackYourProposal/")

wait = WebDriverWait(driver, 20)

advance_btn = wait.until(
    EC.element_to_be_clickable((By.XPATH, "//button[contains(., 'Show Advance Search')]"))
)
driver.execute_script("arguments[0].click();", advance_btn)

# -----------------------------
# Select Major Clearance Type
# -----------------------------
major_clearance = wait.until(
    EC.presence_of_element_located((By.XPATH, "//select[@formcontrolname='majorClearanceType']"))
)
Select(major_clearance).select_by_value("1")

dropdown_element = driver.find_element(By.CSS_SELECTOR, "select[formcontrolname='issueAuthority']")
Select(dropdown_element).select_by_value("SEIAA")

all_table_data = []
headers = []  

# Helper function to trigger Angular change events cleanly
def trigger_angular_select(element, value):
    select_obj = Select(element)
    select_obj.select_by_value(value)
    # Fire 'change' and 'input' events so Angular state updates
    driver.execute_script(
        "arguments[0].dispatchEvent(new Event('change', { bubbles: true }));"
        "arguments[0].dispatchEvent(new Event('input', { bubbles: true }));",
        element
    )

# -----------------------------
# Loop by Static States
# -----------------------------
for state_value, state_name in STATES.items():
    state_dropdown = wait.until(
        EC.presence_of_element_located((By.XPATH, "//select[@formcontrolname='state']"))
    )
    
    # Trigger selection with Angular event dispatch
    trigger_angular_select(state_dropdown, state_value)
    time.sleep(0.5)
    
    print(f"\n🌍 Processing state: {state_name} (Value: {state_value})...")

    search_button = wait.until(
        EC.element_to_be_clickable((By.XPATH, "//button[@type='submit' and contains(.,'Search')]"))
    )
    
    # Store old table reference before searching
    existing_tables = driver.find_elements(By.ID, "excel-table")
    old_table = existing_tables[0] if existing_tables else None

    driver.execute_script("arguments[0].click();", search_button)
    print(f"Search triggered for: {state_name}")
    
    if old_table:
        try:
            wait.until(EC.staleness_of(old_table))
        except Exception:
            time.sleep(1)

    try:
        wait.until(EC.visibility_of_element_located((By.ID, "excel-table")))
        time.sleep(1)  # Allow Angular to finish populating rows
        print(f"Table loaded for state: {state_name}")
    except TimeoutException:
        print(f"⚠️ No results found (Timeout) for state: {state_name}. Skipping...")
        continue

    # -----------------------------
    # PAGINATION SCRAPING LOGIC
    # -----------------------------
    page_num = 1
    
    while True:
        # Get dynamic snapshot of current DOM table
        rows_elements = driver.find_elements(By.XPATH, "//table[@id='excel-table']/tbody/tr")
        
        if not rows_elements:
            print(f"ℹ️ No row elements found for {state_name}. Exiting state.")
            break

        first_row_text = rows_elements[0].text.lower()
        if "no record" in first_row_text or "no data" in first_row_text:
            print(f"ℹ️ Empty state notice detected for {state_name}.")
            break

        # Capture headers once on first pass
        if not headers:
            html = driver.page_source
            soup = BeautifulSoup(html, 'html.parser')
            table = soup.find("table", {"id": "excel-table"})
            if table and table.find("thead"):
                for th in table.find("thead").find_all("th"):
                    headers.append(th.get_text(strip=True))

        total_rows = len(rows_elements)
        print(f"  📊 Page {page_num}: Extracting {total_rows} proposals for {state_name}...")

        # Directly harvest text from live WebElements to avoid Beautifulsoup index drift
        for row in rows_elements:
            try:
                cols = row.find_elements(By.TAG_NAME, "td")
                row_data = [col.text.strip() for col in cols]
                
                if not row_data or len(row_data) <= 1:
                    continue
                
                row_data.append(state_name)
                all_table_data.append(row_data)
            except Exception as e:
                continue

        # -----------------------------
        # Pagination Handling
        # -----------------------------
        try:
            next_buttons = driver.find_elements(By.XPATH, "//button[@aria-label='Next page']")
            if not next_buttons:
                print("  -> Next page button not present. Ending state.")
                break
                
            next_btn = next_buttons[0]
            
            # Check disabled status across standard HTML & Angular attributes
            is_disabled = (
                next_btn.get_attribute("disabled") in ["true", "disabled", True] or
                next_btn.get_attribute("aria-disabled") == "true" or
                "mat-button-disabled" in (next_btn.get_attribute("class") or "")
            )
            
            if is_disabled:
                print(f"  🎉 Completed all pages ({page_num}) for {state_name}!")
                break

            # Capture current row 1 unique signature to verify page transition
            current_signature = rows_elements[0].text if rows_elements else ""

            # Execute pagination click
            driver.execute_script("arguments[0].click();", next_btn)
            page_num += 1

            # Wait for row 1 to update or refresh
            def page_has_updated(d):
                try:
                    new_rows = d.find_elements(By.XPATH, "//table[@id='excel-table']/tbody/tr")
                    if not new_rows:
                        return False
                    return new_rows[0].text != current_signature
                except Exception:
                    return False

            WebDriverWait(driver, 20).until(page_has_updated)
            time.sleep(0.5)

        except TimeoutException:
            print(f"  ⚠️ Page transition timed out on page {page_num}. Ending pagination for state.")
            break
        except Exception as ex:
            print(f"  ⚠️ Pagination exception on page {page_num}: {ex}")
            break

# -----------------------------
# Save Results
# -----------------------------
if all_table_data:
    expected_header_count = len(all_table_data[0])
    
    if len(headers) < expected_header_count:
        headers.append("State_Name")
        
    df = pd.DataFrame(all_table_data, columns=headers[:expected_header_count])
    print("\n--- Final Extracted Dataset Preview ---")
    print(df.head()) 

    output_dir = r"F:\Chimney Work\Marketing\Parivesh Work\Data Architecture\Silver"
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    file_path = os.path.join(output_dir, "SEIAA_KA.xlsx")
    df.to_excel(file_path, index=False)
    print(f"\n✅ Scraped {len(df)} total rows. File saved successfully at:\n{file_path}")
else:
    print("\n❌ Automation complete. Zero data entries found across processed states.")

driver.quit()


🌍 Processing State: KARNATAKA (Value: 29)...
  └── ⚙️ Searching Activity ID: 29 (7(a) Airports...)
  📊 Page 1: Scraping 3 records...
  🎉 Reached final page (1) for Activity 29.
  └── ⚙️ Searching Activity ID: 15 (4(c) Asbestos milling / asbest...)
  ℹ️ No results found for State: 29 + Activity: 15. Proceeding...
  └── ⚙️ Searching Activity ID: 33 (7(da) Bio-Medical Waste Treatm...)
  📊 Page 1: Scraping 10 records...
  📊 Page 2: Scraping 2 records...
  🎉 Reached final page (2) for Activity 33.
  └── ⚙️ Searching Activity ID: 39 (8(a) Building / Construction...)
  📊 Page 1: Scraping 10 records...
  📊 Page 2: Scraping 10 records...
  📊 Page 3: Scraping 10 records...
  📊 Page 4: Scraping 10 records...
  📊 Page 5: Scraping 10 records...
  📊 Page 6: Scraping 10 records...
  📊 Page 7: Scraping 10 records...
  📊 Page 8: Scraping 10 records...
  📊 Page 9: Scraping 10 records...
  📊 Page 10: Scraping 10 records...
  📊 Page 11: Scraping 10 records...
  📊 Page 12: Scraping 10 records...
  📊 Page 